# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [29]:
df = pd.read_csv('data/AviationData.csv', encoding='latin-1')
df.head()

c:\Users\admin\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3145: DtypeWarning: Columns (6,7,28) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.9222,-81.8781,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [30]:
df = df[df['Aircraft.Category'] == 'Airplane'] #this would help us to focus on airplane accidents only

df = df[df['Amateur.Built']!= 'Yes'] #this would help us to focus on commercial airplane accidents only by filtering out amateur built airplanes

df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1986] #this would help us to focus on airplane accidents that happened after 1986(40 yrs)

df.info() #it would basically give me the amount of data we have after filtering. 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21440 entries, 14259 to 88886
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                21440 non-null  object        
 1   Investigation.Type      21440 non-null  object        
 2   Accident.Number         21440 non-null  object        
 3   Event.Date              21440 non-null  datetime64[ns]
 4   Location                21433 non-null  object        
 5   Country                 21439 non-null  object        
 6   Latitude                19173 non-null  object        
 7   Longitude               19167 non-null  object        
 8   Airport.Code            14059 non-null  object        
 9   Airport.Name            14085 non-null  object        
 10  Injury.Severity         20627 non-null  object        
 11  Aircraft.damage         20211 non-null  object        
 12  Aircraft.Category       21440 non-null  ob

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [31]:
injury_columns = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_columns] = df[injury_columns].fillna(0) #this would help us to fill the missing values in the injury columns with 0

df['Total_People'] = df['Total.Fatal.Injuries'] +df['Total.Serious.Injuries'] + df['Total.Minor.Injuries'] + df['Total.Uninjured']

df['Injury_Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total_People']

df[['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total_People', 'Injury_Fraction']].head() #this allows to preview the new columns 

,Total.Fatal.Injuries,Total.Serious.Injuries,Total_People,Injury_Fraction
14259,3.0,2.0,5.0,1.000000
14357,0.0,0.0,2.0,0.000000
14420,2.0,0.0,2.0,1.000000
14421,2.0,0.0,2.0,1.000000
14711,2.0,0.0,3.0,0.666667


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [32]:
#checking for unique values in the column + the missing ones. 
print("Original values in Aircraft.damage:")
print(df['Aircraft.damage'].value_counts(dropna=False))
print("--------------------------------------------------------------------")

#creating a column that shows 1 if the plan was destroyed and 0 if it wasnt. 
df['Is_Destroyed'] = df['Aircraft.damage'].apply(lambda x: 1 if x == 'Destroyed' else 0)

print("Preview of our new tracking column:")
df[['Aircraft.damage', 'Is_Destroyed']].head() #this allows to preview the new column we created.

Original values in Aircraft.damage:
Substantial    16985
Destroyed       2311
NaN             1229
Minor            818
Unknown           97
Name: Aircraft.damage, dtype: int64
--------------------------------------------------------------------
Preview of our new tracking column:


,Aircraft.damage,Is_Destroyed
14259,Destroyed,1
14357,Substantial,0
14420,Destroyed,1
14421,Destroyed,1
14711,Destroyed,1


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [33]:
make_col = 'Make' if 'Make' in df.columns else 'make' # detecting the correct column name for 'Make' 

print(f"Values in {make_col}:")
print(df[make_col].value_counts(dropna=False).head(10)) #this would help us to see the top 10 most common airplane manufacturers in the dataset.
print("--------------------------------------------------------------------")

df[make_col] = df[make_col].astype(str).str.upper().str.strip()#this would help us to standardize the airplane manufacturers names by converting them to uppercase.

print("Top 10 Cleaned values:")
print(df[make_col].value_counts().head(10)) #this would help us to see the top 10 most common airplane manufacturers in the dataset after cleaning.
      

Values in Make:
CESSNA                4867
PIPER                 2803
Cessna                2273
Piper                 1186
BOEING                1037
BEECH                 1018
Beech                  412
MOONEY                 238
Boeing                 231
CIRRUS DESIGN CORP     218
Name: Make, dtype: int64
--------------------------------------------------------------------
Top 10 Cleaned values:
CESSNA                7140
PIPER                 3989
BEECH                 1430
BOEING                1268
MOONEY                 363
AIRBUS                 244
CIRRUS DESIGN CORP     220
AIR TRACTOR INC        219
BELLANCA               219
MAULE                  215
Name: Make, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [36]:
#detect if 'Model' or 'model'
model_col = 'Model' if 'Model' in df.columns else 'model' # detecting the correct column name for 'Model'

#printing top unique values before cleaninng to see raw formatting 
print(f"Values in '{model_col}':")
print(df[model_col].value_counts().head(10)) #this would help us to see the top 10 most common airplane models in the dataset before cleaning.
print("--------------------------------------------------------------------")

#cleaning and converting to string, uppercase and strip spaces
df[model_col] = df[model_col].astype(str).str.upper().str.strip()

print("Top cleaned model values:")
print(df[model_col].value_counts().head(10))

Values in 'Model':
172     772
737     403
152     316
182     304
172S    278
PA28    273
SR22    265
172N    250
180     213
A36     189
Name: Model, dtype: int64
--------------------------------------------------------------------
Top cleaned model values:
172     772
737     403
152     316
182     304
172S    278
PA28    273
SR22    265
172N    250
180     213
A36     189
Name: Model, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [38]:
make_col = 'Make' if 'Make' in df.columns else 'make'
model_col = 'Model' if 'Model' in df.columns else 'model'

df['Aircraft_Type'] = df[make_col] + '' + df[model_col]

print("Preview of new unique identifier column:")
df[[make_col, model_col, 'Aircraft_Type']].head(10)

Preview of new unique identifier column:


,Make,Model,Aircraft_Type
14259,LOCKHEED,L-402-2 (LASA-60),LOCKHEEDL-402-2 (LASA-60)
14357,PIPER,PA-23-250,PIPERPA-23-250
14420,GRUMMAN,G164A,GRUMMANG164A
14421,GRUMMAN,G164A,GRUMMANG164A
14711,CESSNA,TU206F,CESSNATU206F
14712,CESSNA,152,CESSNA152
14805,GRUMMAN,F-14A,GRUMMANF-14A
14806,DOUGLAS,DC-9-31,DOUGLASDC-9-31
16646,MOONEY,M-20C,MOONEYM-20C
16647,SWEARINGEN,SA-226TC,SWEARINGENSA-226TC


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [41]:
df['Injury_Fraction'] = df['Injury_Fraction'].fillna(0) #this would help us to fill the missing values in the injury fraction column with 0

print("Missing values in engineered columns:")
print(df[['Injury_Fraction', 'Is_Destroyed', 'Aircraft_Type']].isnull().sum()) #this would help us to see if there are any missing values in the engineered columns.

Missing values in engineered columns:
Injury_Fraction    0
Is_Destroyed       0
Aircraft_Type      0
dtype: int64


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [42]:
output_file = 'data/cleaned_aviation_data.csv'
df.to_csv(output_file, index=False)
